# Ollama on Colab (Free T4 GPU)

Runs Ollama on a free Google Colab GPU and exposes it via Cloudflare Tunnel.
Paste the printed URL into the endpoint box in your local ollama-ui.

**Before running:** Make sure runtime is set to T4 GPU
`Runtime → Change runtime type → T4 GPU → Save`

In [ ]:
# ── Cell 1: Verify GPU ───────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!apt-get install -y zstd

In [ ]:
# ── Cell 3: Install Ollama ────────────────────────────────────────────────────
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ── Cell 4: Start Ollama server ───────────────────────────────────────────────
import subprocess, time

subprocess.Popen(
    ["ollama", "serve"],
    env={**__import__('os').environ, "OLLAMA_ORIGINS": "*"}
)
time.sleep(3)
print("Ollama running on port 11434")

In [ ]:
# ── Cell 5: Expose via Cloudflare Tunnel (no account needed) ─────────────────
import subprocess, threading, time

!pip install cloudflared -q

public_url = None

def run_tunnel():
    global public_url
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
        stderr=subprocess.PIPE,
        stdout=subprocess.PIPE
    )
    for line in process.stderr:
        line = line.decode()
        if 'trycloudflare.com' in line:
            for part in line.strip().split():
                if 'trycloudflare.com' in part:
                    public_url = part
                    print(f"\n{'='*50}")
                    print(f"Paste this into the endpoint box:")
                    print(f"  {public_url}")
                    print(f"{'='*50}\n")
                    break
            break

threading.Thread(target=run_tunnel, daemon=True).start()
time.sleep(6)

In [ ]:
# ── Cell 6: Pull a model ──────────────────────────────────────────────────────
# Choose one (or run multiple cells for multiple models):
#
#   phi3:mini     — 2.4GB, fast, good general use
#   llama3.2      — 2.0GB, great balance
#   mistral       — 4.1GB, strong reasoning
#   codellama     — 4.1GB, code focused
#   llava         — 4.7GB, multimodal (images)
#   deepseek-r1   — strong reasoning

!ollama pull llama3.2

In [ ]:
# ── Cell 7: List available models ────────────────────────────────────────────
!ollama list

In [ ]:
# ── Cell 8 (optional): Keep session alive ────────────────────────────────────
# Colab disconnects after ~90min of inactivity.
# Run this cell to keep it alive while you work.
import time
print("Keeping session alive... (interrupt to stop)")
while True:
    time.sleep(60)
    print("still alive")